# EDA - Transactions Table

## Setup

In [1]:
import sys
sys.path.append('../')

import polars as pl

import seaborn as sns
import matplotlib.pyplot as plt

from sqlalchemy import create_engine, URL
from utils.db_connection import connection_info

In [2]:
url = URL.create(
    drivername=connection_info['drivername'],
    username=connection_info['username'],
    password=connection_info['password'],
    host=connection_info['host'],
    port=connection_info['port'],
    database="retail"
)

engine = create_engine(url)

conn = engine.connect()

# SELECT * is a horrible idea in production, so I'm splitting up into 3 separate queries based on what type of data they have contained 
# Categorical data, numeric data, and datetime data
categorical_only = pl.read_database(
    query="""
        SELECT 
            transaction_id,
            customer_id,
            payment_method,
            cashier_id,
            return_flag
        FROM transactions
    """,
    connection=conn
)

numeric_only = pl.read_database(
    query="""
        SELECT 
            quantity_sold,
            unit_price_usd,
            discount_rate,
            total_sales_usd,
            shrinkage_usd
        FROM transactions
    """,
    connection=conn
)

datetime_only = pl.read_database(
    query="""
        SELECT 
            transaction_date
        FROM transactions
    """,
    connection=conn
)

conn.close()

## Verification

In [3]:
categorical_only.head(10)

transaction_id,customer_id,payment_method,cashier_id,return_flag
str,str,str,str,str
"""TXN-100000""","""CUST-1265""","""debit card""","""EMP-480""","""no"""
"""TXN-100001""","""CUST-1629""","""debit card""","""EMP-421""","""0"""
"""TXN-100002""","""CUST3283""","""credit card""","""EMP-611""","""y"""
"""TXN-100003""","""CUST-4170""","""CASH""","""EMP-556""","""N"""
"""TXN-100004""","""CUST-3339""","""Credit Card""","""EMP-562""","""N"""
"""TXN-100005""","""CUST-4646""","""Credit Card""","""EMP-663""","""y"""
"""TXN-100006""","""CUST-3378""","""Mobile pay""","""EMP-278""","""Yes"""
"""TXN-100007""","""CUST-5167""","""Gift Card""","""EMP-342""","""no"""
"""TXN-100008""","""CUST-1215""","""credit card""","""EMP-726""","""N"""


In [4]:
categorical_only.null_count()

transaction_id,customer_id,payment_method,cashier_id,return_flag
u32,u32,u32,u32,u32
0,112099,0,0,0


It appears that almost every column is messy aside from the transaction id.

In [5]:
numeric_only.head(10)

quantity_sold,unit_price_usd,discount_rate,total_sales_usd,shrinkage_usd
i64,"decimal[38,2]","decimal[38,4]","decimal[38,2]","decimal[38,2]"
4,1.28,0.0000,5.12,0.00
1,3.12,0.0000,3.12,0.00
1,49.99,0.0000,49.99,0.00
3,12.05,0.0000,36.15,0.00
3,4.84,0.0849,13.29,0.00
1,2.66,0.0647,2.49,0.00
1,11.39,0.0000,11.39,0.00
3,23.79,0.0000,71.37,0.00
1,7.88,0.0000,7.88,0.00


In [6]:
numeric_only.null_count()

quantity_sold,unit_price_usd,discount_rate,total_sales_usd,shrinkage_usd
u32,u32,u32,u32,u32
0,0,0,0,0


Looks good!

In [7]:
datetime_only.head(10)

transaction_date
str
"""2023-01-20 03:50:27"""
"""2023-10-20 08:04:58"""
"""2021-04-22 11:47:11"""
"""2024-08-04 06:14:40"""
"""2023-04-19 15:33:33"""
"""2022-08-21 09:25:48"""
"""2021-12-15 07:41:37"""
"""2023-12-12 06:04:05"""
"""2021-07-24 14:05:38"""


In [8]:
datetime_only.null_count()

transaction_date
u32
0


Little bit of a mess in the only datetime column.

Duplicate rows?

In [9]:
full_df = categorical_only.with_columns(
    numeric_only.select('*')
).with_columns(
    datetime_only.select('*')
)

full_df

transaction_id,customer_id,payment_method,cashier_id,return_flag,quantity_sold,unit_price_usd,discount_rate,total_sales_usd,shrinkage_usd,transaction_date
str,str,str,str,str,i64,"decimal[38,2]","decimal[38,4]","decimal[38,2]","decimal[38,2]",str
"""TXN-100000""","""CUST-1265""","""debit card""","""EMP-480""","""no""",4,1.28,0.0000,5.12,0.00,"""2023-01-20 03:50:27"""
"""TXN-100001""","""CUST-1629""","""debit card""","""EMP-421""","""0""",1,3.12,0.0000,3.12,0.00,"""2023-10-20 08:04:58"""
"""TXN-100002""","""CUST3283""","""credit card""","""EMP-611""","""y""",1,49.99,0.0000,49.99,0.00,"""2021-04-22 11:47:11"""
"""TXN-100003""","""CUST-4170""","""CASH""","""EMP-556""","""N""",3,12.05,0.0000,36.15,0.00,"""2024-08-04 06:14:40"""
"""TXN-100004""","""CUST-3339""","""Credit Card""","""EMP-562""","""N""",3,4.84,0.0849,13.29,0.00,"""2023-04-19 15:33:33"""
…,…,…,…,…,…,…,…,…,…,…
"""TXN-499995""","""CUST-1111""","""Credit Card""","""EMP-998""","""No""",-5,2.84,0.0000,14.20,0.00,"""2022-07-17 20:28:12"""
"""TXN-499996""","""CUST-5467""","""Debit Card""","""EMP-739""","""0""",2,4.24,0.0000,8.48,0.00,"""2021-01-02 17:04:49"""
"""TXN-499997""","""CUST-3365""","""credit card""","""EMP-489""","""N""",1,5.82,0.0000,5.82,0.00,"""2023-09-17 14:29:24"""


In [10]:
full_df.is_duplicated().value_counts()

,count
bool,u32
false,400000


Nope.

## Categorical

In [11]:
categorical_only.select(
    pl.col("transaction_id")
).filter(
    pl.col("transaction_id").str.contains("TXN-[0-9]{6}") == False
)

transaction_id
str


In [12]:
categorical_only.select(
    pl.col("customer_id")
).unique().sort(by='customer_id')

customer_id
str
null
"""CUST-1000"""
"""CUST-1001"""
"""CUST-1002"""
"""CUST-1003"""
…
"""cust-5993"""
"""cust-5994"""
"""cust-5996"""


Well, there are a few lowercase customer_ids that need to be capitalized.

In [13]:
categorical_only.select(
    pl.col("customer_id").value_counts()
).unnest("customer_id").filter(pl.col('customer_id').str.contains("([A-Za-z]{4}-)") == False)

customer_id,count
str,u32
"""CUST5548""",1
"""CUST3405""",1
"""CUST5846""",1
"""CUST4386""",2
"""CUST5749""",2
…,…
"""CUST2051""",5
"""CUST5999""",4
"""CUST2557""",1


And also a missing hypen as well.

In [14]:
categorical_only.select(
    pl.col("customer_id").value_counts()
).unnest("customer_id").filter(pl.col('customer_id').str.contains("([A-Za-z]{4}-|[A-Za-z]{4}[0-9]{4})") == False)

customer_id,count
str,u32
"""N/A""",5745


There's a lot of null values, and plenty of messy values as well...

Maybe I could standardize the customer id first (e.g CUST-0001), then fill in all of the nulls with something like (NULL-0001), increasing sequentially until I reach the max. However, I'm assuming that each null customer is unique which isn't true because there can be a single "null" customer that has multiple transactions. I might just fill all the nulls with "N/A" instead now that I think of it to avoid the problem of having "unique" identifiers.

In [15]:
categorical_only.select(
    pl.col("payment_method")
).unique()

payment_method
str
"""CREDIT CARD"""
"""Credit card"""
"""Cash"""
"""Debit Card"""
"""cash"""
…
"""Gift Card"""
"""mobile pay"""
"""debit card"""


More messy values!

In [16]:
categorical_only.select(
    pl.col("cashier_id").value_counts()
).unnest("cashier_id").filter(pl.col('cashier_id').str.contains("EMP-[0-9]{3}") == False)

cashier_id,count
str,u32


In [17]:
categorical_only.select(
    pl.col("return_flag")
).unique()

return_flag
str
"""0"""
"""No"""
"""1"""
"""N"""
"""y"""
"""no"""
"""Yes"""
"""Y"""


Same issue like the other flag columns. Just need to standardize them.

## Numerical

In [18]:
numeric_only.describe()

statistic,quantity_sold,unit_price_usd,discount_rate,total_sales_usd,shrinkage_usd
str,f64,f64,f64,f64,f64
"""count""",400000.0,400000.0,400000.0,400000.0,400000.0
"""null_count""",0.0,0.0,0.0,0.0,0.0
"""mean""",2.010005,9.847197,0.015936,23.721174,0.059541
"""std""",1.933462,10.902142,0.032786,32.618617,0.208912
"""min""",-5.0,-49.99,0.0,0.39,0.0
"""25%""",1.0,3.68,0.0,6.25,0.0
"""50%""",2.0,7.06,0.0,13.04,0.0
"""75%""",3.0,13.21,0.0131,27.76,0.0
"""max""",20.0,49.99,0.2378,749.85,3.5


Quantity sold and unit price have negative values based on the summary stats. Worth investigating before generating some histograms.

In [19]:
full_df.filter(
    pl.col('quantity_sold') <= 0
)

transaction_id,customer_id,payment_method,cashier_id,return_flag,quantity_sold,unit_price_usd,discount_rate,total_sales_usd,shrinkage_usd,transaction_date
str,str,str,str,str,i64,"decimal[38,2]","decimal[38,4]","decimal[38,2]","decimal[38,2]",str
"""TXN-100030""",null,"""Debit Card""","""EMP-471""","""y""",-2,6.29,0.0000,12.58,0.00,"""??"""
"""TXN-100043""","""CUST-4430""","""debit card""","""EMP-613""","""1""",-2,9.35,0.0000,18.70,0.00,"""2024-03-18 14:52:28"""
"""TXN-100094""","""CUST-3626""","""Gift card""","""EMP-326""","""No""",-5,-10.25,0.0000,51.25,0.00,"""TBD"""
"""TXN-100116""","""CUST-1128""","""Debit Card""","""EMP-117""","""y""",-3,18.65,0.0000,55.95,0.00,"""2024-08-03 12:43:48"""
"""TXN-100132""",null,"""mobile pay""","""EMP-290""","""0""",-4,5.64,0.0000,22.56,0.00,"""2022-09-23 23:09:06"""
…,…,…,…,…,…,…,…,…,…,…
"""TXN-499848""","""CUST-4771""","""Gift Card""","""EMP-273""","""1""",-5,-3.86,0.0000,19.30,0.00,"""2024-02-17 05:26:16"""
"""TXN-499893""",null,"""Debit Card""","""EMP-512""","""Y""",-2,20.31,0.0000,40.62,0.00,"""2022-08-14 05:51:13"""
"""TXN-499904""","""CUST-3425""","""cash""","""EMP-161""","""N""",-5,9.83,0.0000,49.15,0.00,"""2021-02-12 17:37:25"""


In [20]:
full_df.filter(
    pl.col('unit_price_usd') <= 0
)

transaction_id,customer_id,payment_method,cashier_id,return_flag,quantity_sold,unit_price_usd,discount_rate,total_sales_usd,shrinkage_usd,transaction_date
str,str,str,str,str,i64,"decimal[38,2]","decimal[38,4]","decimal[38,2]","decimal[38,2]",str
"""TXN-100019""","""CUST-3006""","""CREDIT CARD""","""EMP-174""","""Y""",1,-2.14,0.0000,2.14,0.00,"""2024-04-28 01:25:35"""
"""TXN-100094""","""CUST-3626""","""Gift card""","""EMP-326""","""No""",-5,-10.25,0.0000,51.25,0.00,"""TBD"""
"""TXN-100149""","""CUST-5619""","""Gift Card""","""EMP-798""","""y""",7,-13.38,0.0000,93.66,0.00,"""2024-06-08 17:40:27"""
"""TXN-100165""","""CUST-2429""","""CREDIT CARD""","""EMP-573""","""N""",1,-3.78,0.0000,3.78,0.00,"""2022-03-03 07:45:12"""
"""TXN-100182""","""CUST-1927""","""cash""","""EMP-655""","""Y""",2,-14.67,0.0000,29.34,0.00,""""""
…,…,…,…,…,…,…,…,…,…,…
"""TXN-499936""",null,"""credit card""","""EMP-467""","""N""",2,-7.96,0.0000,15.92,0.20,"""2023-07-23 20:48:25"""
"""TXN-499953""","""CUST-5094""","""Credit Card""","""EMP-371""","""Yes""",1,-2.90,0.0000,2.90,0.00,"""2021-08-15 00:59:27"""
"""TXN-499959""","""CUST-2539""","""Debit Card""","""EMP-392""","""no""",2,-32.11,0.0000,64.22,0.54,"""2023-08-08 23:32:28"""


Plenty of refunds.

## Datetime

In [21]:
datetime_only.select(
    pl.col("transaction_date")
).filter(
    pl.col("transaction_date").str.contains("[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}") == False
).unique()

transaction_date
str
"""TBD"""
""""""
"""??"""
"""N/A"""


Wow! More messy data! I'll convert these to nulls to maintain consistency.